# 07 One-Click LexAI Runner (Hardened)

This notebook is designed to run end-to-end with robust fallbacks for Databricks workspace notebook paths.

Run order: `Cell 1 -> Cell 10`


In [ ]:
# CELL 1: Runtime Flags
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = False

FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

# Set explicitly if auto-detection fails:
REPO_DIR_OVERRIDE = "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform"

SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
    "REPO_DIR_OVERRIDE": REPO_DIR_OVERRIDE,
})


In [ ]:
# CELL 2: Resolve repo path safely
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_required_files(repo_dir: Path) -> bool:
    return (repo_dir / "apps" / "fastapi_app.py").exists() and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()


def _safe_walk_for_repo(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _filenames in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        p = Path(dirpath)
        try:
            if _repo_has_required_files(p):
                return p
        except Exception:
            continue
    return None


def _context_repo_guess():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        nb_path = ctx.notebookPath().get()  # /Users/<email>/<repo>/notebooks/07_...
        if not nb_path:
            return None
        pp = Path(nb_path)
        # repo likely parent.parent
        ws_repo = Path("/Workspace") / Path(*pp.parent.parent.parts[1:])
        if ws_repo.exists() and _repo_has_required_files(ws_repo):
            return ws_repo
    except Exception:
        pass
    return None


def resolve_repo_dir() -> Path:
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_required_files(p):
            return p

    cwd = Path(os.getcwd()).resolve()
    for cand in [cwd] + list(cwd.parents):
        try:
            if _repo_has_required_files(cand):
                return cand
        except Exception:
            continue

    g = _context_repo_guess()
    if g is not None:
        return g

    for root in [Path("/Workspace/Repos"), Path("/Workspace/Users"), Path("/Workspace")]:
        if not root.exists():
            continue
        hit = _safe_walk_for_repo(root)
        if hit is not None:
            return hit

    raise FileNotFoundError(
        "Could not locate repo root. Set REPO_DIR_OVERRIDE to your repo path."
    )


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

log(f"Repo root: {REPO_DIR}")
print("[CELL 2] OK")


In [ ]:
# CELL 3: Dependency preflight (no Python restart)
import importlib
import importlib.metadata as ilm
import subprocess

REQ_FILE = Path("apps/requirements.txt")
if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

required_packages = [
    "fastapi",
    "uvicorn",
    "streamlit",
    "requests",
    "pydantic",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "mlflow",
    "databricks-sdk",
    "typing_extensions>=4.6.0",
]


def _pkg_name(spec: str) -> str:
    for sep in [">=", "==", "<=", "~=", ">", "<"]:
        if sep in spec:
            return spec.split(sep)[0].strip()
    return spec.strip()


missing_specs = []
for spec in required_packages:
    name = _pkg_name(spec)
    try:
        ilm.version(name)
    except Exception:
        missing_specs.append(spec)

print("[CELL 3] Missing specs:", missing_specs)

if missing_specs and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQ_FILE)] + missing_specs
    print("[CELL 3] Installing missing packages...")
    subprocess.check_call(cmd)
    print("[CELL 3] Installation complete")
elif missing_specs and not AUTO_INSTALL_MISSING:
    raise RuntimeError(f"Missing packages: {missing_specs}. Set AUTO_INSTALL_MISSING=True")

# Hard validation for TypeIs support without forcing restart.
try:
    import typing_extensions
    importlib.reload(typing_extensions)
    _ = typing_extensions.TypeIs
    print("[CELL 3] typing_extensions.TypeIs available")
except Exception as e:
    raise RuntimeError(
        "typing_extensions TypeIs still unavailable. Run: dbutils.library.restartPython(), then rerun from Cell 1"
    ) from e

print("[CELL 3] OK")


In [ ]:
# CELL 4: Spark / cluster context
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print("[CELL 4] Spark session ready:", bool(spark))

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"

for key, var in [
    ("spark.databricks.clusterUsageTags.clusterId", "cluster_id"),
    ("spark.databricks.clusterUsageTags.orgId", "org_id"),
    ("spark.databricks.workspaceUrl", "workspace_url"),
]:
    try:
        val = spark.conf.get(key)
        if var == "cluster_id":
            cluster_id = val
        elif var == "org_id":
            org_id = val
        elif var == "workspace_url":
            workspace_url = val
    except Exception:
        pass

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)


In [ ]:
# CELL 5: Initialize notebook-06 engine through adapter
from pathlib import Path
import importlib
import os
import apps.lexai06_notebook_adapter as _adapter

importlib.reload(_adapter)
NotebookEngine = _adapter.NotebookEngine

workspace_candidates = [
    "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine",
    "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
    str(Path(REPO_DIR) / "notebooks" / "06_High-precision_QA_Legal_Reasoning_Engine.ipynb"),
    str(Path(REPO_DIR) / "apps" / "notebook_06_snapshot.ipynb"),
]

print("[CELL 5] Notebook candidates:")
for c in workspace_candidates:
    try:
        print(" -", c, "exists=", Path(c).exists())
    except Exception:
        print(" -", c, "exists=ERROR")

status = None
last_err = None
for cand in workspace_candidates:
    try:
        os.environ["LEXAI06_NOTEBOOK_PATH"] = cand
        engine = NotebookEngine(notebook_path=Path(cand))
        status = engine.initialize()
        print(f"[CELL 5] Initialized using candidate: {cand}")
        break
    except Exception as e:
        last_err = e
        print(f"[CELL 5] Candidate failed: {cand} -> {e}")

if status is None:
    raise RuntimeError(f"Engine initialization failed for all candidates. Last error: {last_err}")

print("[CELL 5] Engine initialized")
for k, v in status.items():
    print(f"  - {k}: {v}")

if not status.get("ready"):
    raise RuntimeError(f"Engine failed to initialize: {status}")


In [ ]:
# CELL 6: Smoke test (optional)
if RUN_SMOKE_TEST:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] RUN_SMOKE_TEST=False -> skipped")


In [ ]:
# CELL 7: Start FastAPI server (background thread)
import threading
import uvicorn

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")

if START_FASTAPI:
    if FASTAPI_THREAD is not None and FASTAPI_THREAD.is_alive():
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")
    else:
        from apps.fastapi_app import app
        config = uvicorn.Config(app, host="0.0.0.0", port=int(FASTAPI_PORT), log_level="info")
        FASTAPI_SERVER = uvicorn.Server(config)
        FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
        FASTAPI_THREAD.start()
        globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
        globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
        print(f"[CELL 7] FastAPI started on 0.0.0.0:{FASTAPI_PORT}")

    print("[CELL 7] Local health URL:", f"http://127.0.0.1:{FASTAPI_PORT}/health")

    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        proxy_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{FASTAPI_PORT}/health"
        print("[CELL 7] Driver proxy URL:", proxy_url)
else:
    print("[CELL 7] START_FASTAPI=False -> skipped")


In [ ]:
# CELL 8: FastAPI smoke call (optional)
import requests

if START_FASTAPI:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] /v1/legal/answer status:", r.status_code)
        try:
            body = r.json()
            print("[CELL 8] answer preview:", body.get("answer", "")[:500])
        except Exception:
            print("[CELL 8] raw response:", r.text[:500])
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] START_FASTAPI=False -> skipped")


In [ ]:
# CELL 9: Optional Streamlit start (blocking)
import subprocess

if START_STREAMLIT:
    os.environ["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "apps/streamlit_app.py",
        "--server.port", str(STREAMLIT_PORT),
        "--server.address", "0.0.0.0",
    ]
    print("[CELL 9] Starting Streamlit:", " ".join(cmd))
    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        ui_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{STREAMLIT_PORT}/"
        print("[CELL 9] Streamlit URL:", ui_url)
    subprocess.call(cmd)
else:
    print("[CELL 9] START_STREAMLIT=False -> skipped")


In [ ]:
# CELL 10: Stop helper
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")
